# Merge Atribut ke floods.csv Final

Notebook ini menggabungkan output dari Notebook 5 (klasifikasi jalan) dan
Notebook 6 (jarak faskes OSRM) ke dalam satu file `floods.csv` yang siap
dipakai backend. Jalankan notebook ini **setelah** NB 5 dan NB 6 selesai,
dan **sebelum** NB 4 (matriks jarak OSRM).

**Input**
- `skenario_*.csv` (output NB 7, skenario terpilih)
- `road_classification_surabaya.csv` (output NB 5)
- `faskes_distance_report.csv` (output NB 6)

**Output**
- `floods_final.csv` dengan kolom lengkap yang dibutuhkan backend


In [1]:
import pandas as pd
from pathlib import Path

# Output NB 3: file skenario terpilih
SCENARIO_FILE = "../backend/app/data/floods-skenario1.csv"

# Output NB 4: klasifikasi jalan
ROAD_CLASS_FILE = "output_road_class/road_classification_surabaya.csv"

# Output NB 5: jarak faskes OSRM
FASKES_DIST_FILE = "output_faskes_dist/faskes_distance_report.csv"

OUTPUT_DIR = Path("output_final")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup selesai.")
print(f"Skenario: {SCENARIO_FILE}")

Setup selesai.
Skenario: ../backend/app/data/floods-skenario1.csv


## 1. Muat skenario terpilih

In [2]:
floods = pd.read_csv(SCENARIO_FILE)
print(f"Titik genangan: {len(floods)}")
print(f"Kolom: {list(floods.columns)}")

# Pastikan ada kolom id untuk merge
if "no" in floods.columns and "id" not in floods.columns:
    floods["id"] = [f"F_{i:04d}" for i in range(len(floods))]
elif "id" not in floods.columns:
    floods["id"] = [f"F_{i:04d}" for i in range(len(floods))]

print(f"Preview:")
floods[["id", "jalan", "kecamatan", "depth_cm", "lat", "lon"]].head()

Titik genangan: 34
Kolom: ['no', 'tanggal', 'jalan', 'kecamatan', 'kelurahan', 'area', 'detail', 'depth_cm', 'lat', 'lon', 'terima_berita', 'tiba_tkp', 'mulai_penanganan', 'selesai']
Preview:


,id,jalan,kecamatan,depth_cm,lat,lon
0,F_0000,SWK Ketintang dan Jl. Lusi,Gayungan,5.0,-7.316784,112.729173
1,F_0001,Wisma Tengger,Benowo,10.0,-7.262527,112.656512
2,F_0002,SMPN 12,Wonokromo,20.0,-7.297865,112.749232
3,F_0003,JL. Bung Tomo Ngagel dan Jl. Ngagel Timur,Wonokromo,20.0,-7.290469,112.745126
4,F_0004,Gereja SMTB,Gubeng,5.0,-7.288979,112.760230


## 2. Merge klasifikasi jalan (NB 5)

In [3]:
road_class = pd.read_csv(ROAD_CLASS_FILE)
print(f"Road classification records: {len(road_class)}")
print(f"Kolom: {list(road_class.columns)}")

# Merge berdasarkan flood_id
floods = floods.merge(
    road_class[["flood_id", "road_class", "highway_type_used", "distance_to_road_m"]],
    left_on="id",
    right_on="flood_id",
    how="left"
).drop(columns=["flood_id"], errors="ignore")

n_road_ok = floods["road_class"].notna().sum()
n_road_miss = floods["road_class"].isna().sum()
print(f"road_class terisi: {n_road_ok}, kosong: {n_road_miss}")

if n_road_miss > 0:
    print(f"  -> Titik tanpa road_class akan di-default ke 3 (tertiary)")
    floods["road_class"] = floods["road_class"].fillna(3).astype(int)

Road classification records: 34
Kolom: ['flood_id', 'road_class', 'highway_type_used', 'distance_to_road_m', 'within_search_radius']
road_class terisi: 34, kosong: 0


## 3. Merge jarak faskes OSRM (NB 6)

In [4]:
faskes_dist = pd.read_csv(FASKES_DIST_FILE)
print(f"Faskes distance records: {len(faskes_dist)}")
print(f"Kolom: {list(faskes_dist.columns)}")

# Merge berdasarkan flood_id
floods = floods.merge(
    faskes_dist[["flood_id", "dist_faskes_osrm_m", "nearest_faskes_name"]],
    left_on="id",
    right_on="flood_id",
    how="left"
).drop(columns=["flood_id"], errors="ignore")

# Rename ke kolom yang diharapkan backend
floods = floods.rename(columns={"dist_faskes_osrm_m": "dist_faskes_m"})

n_faskes_ok = floods["dist_faskes_m"].notna().sum()
n_faskes_miss = floods["dist_faskes_m"].isna().sum()
print(f"dist_faskes_m terisi: {n_faskes_ok}, kosong: {n_faskes_miss}")

if n_faskes_miss > 0:
    median_dist = floods["dist_faskes_m"].median()
    print(f"  -> Titik tanpa dist_faskes_m akan di-default ke median ({median_dist:.0f} m)")
    floods["dist_faskes_m"] = floods["dist_faskes_m"].fillna(median_dist)

Faskes distance records: 34
Kolom: ['flood_id', 'nearest_faskes_name', 'nearest_faskes_idx', 'dist_faskes_osrm_m', 'dist_faskes_haversine_m', 'ratio_osrm_hav', 'fallback_haversine']
dist_faskes_m terisi: 34, kosong: 0


## 4. Format ke skema backend dan simpan

In [5]:
# Backend mengharapkan kolom dengan nama ini (lihat main.py _load_floods)
floods_final = pd.DataFrame({
    "Datetime": floods.get("tanggal", ""),
    "Latitude": floods["lat"],
    "Longitude": floods["lon"],
    "Deskripsi": floods.get("detail", floods.get("jalan", "")),
    "Images": "",  # tidak ada gambar dari data Damkar
    "Ketinggian (cm)": floods["depth_cm"],
    "road_class": floods["road_class"].astype(int),
    "dist_faskes_m": floods["dist_faskes_m"].round(2),
})

print("=== floods_final ===")
print(f"Shape: {floods_final.shape}")
print(f"Kolom: {list(floods_final.columns)}")
print()

# Validasi
assert floods_final["Latitude"].notna().all(), "Ada Latitude kosong!"
assert floods_final["Longitude"].notna().all(), "Ada Longitude kosong!"
assert floods_final["Ketinggian (cm)"].notna().all(), "Ada Ketinggian kosong!"
assert floods_final["road_class"].notna().all(), "Ada road_class kosong!"
assert floods_final["dist_faskes_m"].notna().all(), "Ada dist_faskes_m kosong!"
print("Validasi OK: semua kolom wajib terisi.")
print()

# Simpan
floods_final.to_csv(OUTPUT_DIR / "floods_final.csv", index=False)
print(f"Tersimpan: {OUTPUT_DIR}/floods_final.csv")
print()
print("Langkah berikutnya:")
print("1. Copy floods_final.csv ke backend/app/data/floods.csv")
print("2. Jalankan NB 4 (matriks OSRM) dengan floods.csv baru ini")
print("3. Copy distance_matrix.npy dan time_matrix.npy ke backend/app/data/")
print("4. Push dan deploy")

floods_final.head(10)

=== floods_final ===
Shape: (34, 8)
Kolom: ['Datetime', 'Latitude', 'Longitude', 'Deskripsi', 'Images', 'Ketinggian (cm)', 'road_class', 'dist_faskes_m']

Validasi OK: semua kolom wajib terisi.

Tersimpan: output_final/floods_final.csv

Langkah berikutnya:
1. Copy floods_final.csv ke backend/app/data/floods.csv
2. Jalankan NB 4 (matriks OSRM) dengan floods.csv baru ini
3. Copy distance_matrix.npy dan time_matrix.npy ke backend/app/data/
4. Push dan deploy


,Datetime,Latitude,Longitude,Deskripsi,Images,Ketinggian (cm),road_class,dist_faskes_m
0,2026-06-22,-7.316784,112.729173,Genangan : 5 cm >> Surut>>,,5.0,2,171.3
1,2026-06-22,-7.262527,112.656512,Genangan : 10 cm = Surut Clear 0%,,10.0,2,465.0
2,2026-06-22,-7.297865,112.749232,Genangan : 20 cm,,20.0,4,1093.5
3,2026-06-22,-7.290469,112.745126,Genangan : 20 cm,,20.0,5,518.2
4,2026-06-22,-7.288979,112.760230,Genangan : 5 cm,,5.0,3,105.4
5,2026-06-22,-7.296059,112.759458,Tinggi Genangan : 30 cm,,30.0,3,146.9
6,2026-06-22,-7.265701,112.712539,Genangan : 30 cm,,30.0,2,421.4
7,2026-06-22,-7.272245,112.705659,Genangan : 30 cm,,30.0,2,257.6
8,2026-06-22,-7.312167,112.761497,Genangan : 15 cm,,15.0,2,1926.8
9,2026-06-22,-7.325332,112.745502,Genangan : 5 cm,,5.0,4,0.0


## 5. Ringkasan statistik skenario

In [ ]:
print("=== RINGKASAN SKENARIO ===")
print(f"Jumlah titik genangan: {len(floods_final)}")
print()
print("Kedalaman air (cm):")
print(f"  Min    : {floods_final['Ketinggian (cm)'].min()}")
print(f"  Median : {floods_final['Ketinggian (cm)'].median()}")
print(f"  Max    : {floods_final['Ketinggian (cm)'].max()}")
print(f"  Mean   : {floods_final['Ketinggian (cm)'].mean():.1f}")
print()
print("Klasifikasi jalan:")
print(floods_final["road_class"].value_counts().sort_index().to_string())
print()
print("Jarak ke faskes terdekat (m):")
print(f"  Min    : {floods_final['dist_faskes_m'].min():.0f}")
print(f"  Median : {floods_final['dist_faskes_m'].median():.0f}")
print(f"  Max    : {floods_final['dist_faskes_m'].max():.0f}")
print(f"  Mean   : {floods_final['dist_faskes_m'].mean():.0f}")

=== RINGKASAN SKENARIO ===
Jumlah titik genangan: 34

Kedalaman air (cm):
  Min    : 5.0
  Median : 20.0
  Max    : 50.0
  Mean   : 22.6

Klasifikasi jalan:
road_class
1     3
2    12
3    10
4     6
5     3

Jarak ke faskes terdekat (m):
  Min    : 0
  Median : 492
  Max    : 1927
  Mean   : 657


: 

## Catatan

- Notebook ini **idempoten**: bisa dijalankan ulang tanpa efek samping selama
  input file tidak berubah.
- Jika NB 5 atau NB 6 belum dijalankan (file output belum ada), notebook
  ini akan error di langkah merge. Pastikan urutan eksekusi sesuai pipeline.
- Statistik di langkah 5 berguna untuk narasi di Bab 4.1 (deskripsi data)
  dan Bab 6 (konteks pembahasan hasil).
